# Assignment 2 – Bank Marketing classification
Train 5 classifiers on `bank.csv` (predict term-deposit subscription `y`), compare them, and save the fitted pipelines.

## 1. Dataset

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, f1_score, matthews_corrcoef,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# Project root = the folder this notebook lives in
ROOT = Path(".").resolve()
MODEL_DIR = ROOT / "model"
MODEL_DIR.mkdir(exist_ok=True)

TARGET = "y"
POS_LABEL = 1  # 'yes' (client subscribed a term deposit)

In [2]:
# bank.csv is semicolon-separated
df = pd.read_csv(ROOT / "bank.csv", sep=";")

n_features = df.shape[1] - 1
assert n_features >= 12, f"Need >= 12 features, got {n_features}"
assert df.shape[0] >= 500, f"Need >= 500 instances, got {df.shape[0]}"
print(f"Shape: {df.shape}  ({n_features} features, {df.shape[0]} instances)  -> requirements met")

print("\nDtypes:")
print(df.dtypes.to_string())

print("\nMissing values per column:")
print(df.isna().sum().to_string())

print("\nClass balance (raw target):")
print(pd.concat([df[TARGET].value_counts(), df[TARGET].value_counts(normalize=True).round(4)], axis=1).to_string())

Shape: (4521, 17)  (16 features, 4521 instances)  -> requirements met

Dtypes:
age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object

Missing values per column:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0

Class balance (raw target):
     count  proportion
y                     
no    4000      0.8848
yes    521      0.1152


In [3]:
# Encode target: yes -> 1 (positive class), no -> 0
df[TARGET] = df[TARGET].map({"no": 0, "yes": 1})
assert df[TARGET].notna().all()

X = df.drop(columns=TARGET)
y = df[TARGET]

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]
print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

Numeric: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Categorical: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


## 2. Stratified train/test split (80/20)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Positive rate  train: %.4f  test: %.4f" % (y_train.mean(), y_test.mean()))

Train: (3616, 16)  Test: (905, 16)
Positive rate  train: 0.1153  test: 0.1149


## 3. Preprocessing and pipelines

In [5]:
def make_preprocessor():
    # sparse_output=False: GaussianNB cannot consume sparse matrices
    return ColumnTransformer([
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ])

classifiers = {
    "logistic_regression": LogisticRegression(max_iter=1000),
    "decision_tree": DecisionTreeClassifier(random_state=42),
    "knn": KNeighborsClassifier(),
    "naive_bayes": GaussianNB(),
    "svm": SVC(probability=True, random_state=42),
}

# One independent Pipeline per model, fit on the training split only
pipelines = {
    name: Pipeline([("preprocess", make_preprocessor()), ("clf", clf)])
    for name, clf in classifiers.items()
}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    print("fitted", name)

fitted logistic_regression
fitted decision_tree
fitted knn
fitted naive_bayes
fitted svm


## 4. Metrics

In [6]:
display_names = {
    "logistic_regression": "Logistic Regression",
    "decision_tree": "Decision Tree",
    "knn": "KNN",
    "naive_bayes": "Naive Bayes",
    "svm": "SVM",
}

rows = {}
for name, pipe in pipelines.items():
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    rows[display_names[name]] = {
        "Accuracy": accuracy_score(y_test, pred),
        "AUC": roc_auc_score(y_test, proba),
        "Precision": precision_score(y_test, pred, average="binary", pos_label=POS_LABEL),
        "Recall": recall_score(y_test, pred, average="binary", pos_label=POS_LABEL),
        "F1": f1_score(y_test, pred, average="binary", pos_label=POS_LABEL),
        "MCC": matthews_corrcoef(y_test, pred),
    }

metrics = pd.DataFrame.from_dict(rows, orient="index")
metrics.index.name = "Model"
assert not metrics.isna().any().any(), "NaN found in metrics"
metrics.round(4)

,Accuracy,AUC,Precision,Recall,F1,MCC
Model,,,,,,
Logistic Regression,0.8928,0.8913,0.5636,0.2981,0.3899,0.3579
Decision Tree,0.8586,0.6942,0.4032,0.4808,0.4386,0.3602
KNN,0.8840,0.7692,0.4872,0.1827,0.2657,0.2477
Naive Bayes,0.8210,0.7881,0.3092,0.4519,0.3672,0.2737
SVM,0.8939,0.8830,0.5909,0.2500,0.3514,0.3374


## 5. Save artefacts

In [7]:
test_df = X_test.copy()
test_df[TARGET] = y_test
test_df.to_csv(ROOT / "test_data.csv", index=False)

metrics.round(4).to_csv(ROOT / "metrics_summary.csv")

for name, pipe in pipelines.items():
    joblib.dump(pipe, MODEL_DIR / f"{name}.joblib")

print("test_data.csv:", test_df.shape, "| has target:", TARGET in test_df.columns)
print("metrics_summary.csv saved")
print("models:", sorted(p.name for p in MODEL_DIR.glob("*.joblib")))

test_data.csv: (905, 17) | has target: True
metrics_summary.csv saved
models: ['decision_tree.joblib', 'knn.joblib', 'logistic_regression.joblib', 'naive_bayes.joblib', 'svm.joblib']


## 6. Final comparison

In [8]:
print(metrics.round(4).to_string())

                     Accuracy     AUC  Precision  Recall      F1     MCC
Model                                                                   
Logistic Regression    0.8928  0.8913     0.5636  0.2981  0.3899  0.3579
Decision Tree          0.8586  0.6942     0.4032  0.4808  0.4386  0.3602
KNN                    0.8840  0.7692     0.4872  0.1827  0.2657  0.2477
Naive Bayes            0.8210  0.7881     0.3092  0.4519  0.3672  0.2737
SVM                    0.8939  0.8830     0.5909  0.2500  0.3514  0.3374
